In [ ]:
from queue import PriorityQueue

class Node:
    def __init__(self, position, parent=None):
        self.position = position
        self.parent = parent
        self.g = 0
        self.h = 0
        self.f = 0

    def __lt__(self, other):
        return self.f < other.f

def heuristic(current_pos, end_pos):
    return abs(current_pos[0] - end_pos[0]) + abs(current_pos[1] - end_pos[1])

def best_first_search(maze, start, end):
    rows, cols = len(maze), len(maze[0])
    start_node = Node(start)
    frontier = PriorityQueue()
    frontier.put(start_node)
    visited = set()

    while not frontier.empty():
        current_node = frontier.get()
        current_pos = current_node.position

        if current_pos == end:
            path = []
            while current_node:
                path.append(current_node.position)
                current_node = current_node.parent
            return path[::-1]

        visited.add(current_pos)

        for dx, dy in [(1, 0), (-1, 0), (0, 1), (0, -1)]:
            new_pos = (current_pos[0] + dx, current_pos[1] + dy)
            if 0 <= new_pos[0] < rows and 0 <= new_pos[1] < cols and maze[new_pos[0]][new_pos[1]] == 0 and new_pos not in visited:
                new_node = Node(new_pos, current_node)
                new_node.h = heuristic(new_pos, end)
                new_node.f = new_node.h  
                frontier.put(new_node)
                visited.add(new_pos)
    return None

def find_full_path(maze, start, goals):
    current_start = start
    full_path = []
    
    for goal in goals:
        print(f"Searching: {current_start} -> {goal}")
        segment = best_first_search(maze, current_start, goal)
        
        if segment:
            if not full_path:
                full_path.extend(segment)
            else:
                full_path.extend(segment[1:])
            current_start = goal
        else:
            print(f"Cannot reach goal {goal}!")
            return None
    return full_path

maze = [
    [0, 0, 1, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 0, 1, 0, 1],
    [0, 0, 1, 0, 0],
    [0, 0, 0, 1, 0]
]

start = (0, 0)
end = [(1, 1), (0, 4), (4, 4)] 

path = find_full_path(maze, start, end)

if path:
    print("\nFinal Path covering all goals:")
    print(path)

Searching: (0, 0) -> (1, 1)
Searching: (1, 1) -> (0, 4)
Searching: (0, 4) -> (4, 4)

Final Path covering all goals:
[(0, 0), (1, 0), (1, 1), (1, 2), (1, 3), (0, 3), (0, 4), (1, 4), (1, 3), (2, 3), (3, 3), (3, 4), (4, 4)]


In [8]:
import heapq
import random
import time

map_data = {
    'H': {'S': 5, 'P': 2},
    'S': {'O': 10, 'M': 3},
    'P': {'M': 8, 'G': 4},
    'G': {'O': 6},
    'M': {'O': 4},
    'O': {}}

h_scores = {
    'H': 15, 'S': 10, 'P': 12, 
    'M': 4, 'G': 6, 'O': 0
}

def fluctuate_costs(network):
    for loc in network:
        for adj in network[loc]:
            adjustment = random.randint(-3, 3)
            network[loc][adj] = max(1, network[loc][adj] + adjustment)
    return network

def navigate_map(network, origin, destination):
    open_list = [(h_scores[origin], origin)]
    backtrack = {origin: None}
    actual_costs = {origin: 0}
    explored = set()

    while open_list:
        heapq.heapify(open_list)
        _, active_node = heapq.heappop(open_list)

        if active_node == destination:
            route = []
            while active_node:
                route.append(active_node)
                active_node = backtrack[active_node]
            return route[::-1]

        explored.add(active_node)

        for neighbor, weight in network[active_node].items():
            if neighbor in explored:
                continue
            
            tentative_g = actual_costs[active_node] + weight
            
            if neighbor not in actual_costs or tentative_g < actual_costs[neighbor]:
                actual_costs[neighbor] = tentative_g
                f_total = tentative_g + h_scores[neighbor]
                backtrack[neighbor] = active_node
                heapq.heappush(open_list, (f_total, neighbor))
    return None

def execute_dynamic_search(network, start_node, end_node, cycles=3):
    print("starting adaptive A* navigation:")
    traversed_path = []
    agent_location = start_node

    for i in range(1, cycles + 1):
        print(f"\n(cycle {i}) monitoring network fluctuations")
        optimal_segment = navigate_map(network, agent_location, end_node)

        if not optimal_segment:
            print("error: route obstructed")
            break

        if not traversed_path:
            traversed_path.extend(optimal_segment)
        else:
            traversed_path.extend(optimal_segment[1:])

        print(f"Agent currently at: {agent_location}")
        print(f"Calculated Segment: {' -> '.join(optimal_segment)}")

        if optimal_segment[-1] == end_node:
            print(f"\nsuccess, destination reached")
            print(f"final path: {traversed_path}")
            return 

        agent_location = optimal_segment[min(1, len(optimal_segment)-1)]
        network = fluctuate_costs(network)
        time.sleep(0.5)

    return traversed_path

execute_dynamic_search(map_data, 'H', 'O', cycles=4)

starting adaptive A* navigation:

(cycle 1) monitoring network fluctuations
Agent currently at: H
Calculated Segment: H -> P -> G -> O

success, destination reached
final path: ['H', 'P', 'G', 'O']


In [20]:
import heapq

delivery_points = {
    'Start': {'A': 2, 'B': 5},
    'A': {'D': 8},
    'B': {'D': 3, 'E': 6},
    'C': {'Goal': 5},
    'D': {'C': 2},
    'E': {'D': 4},
    'Goal': {}
}

deadlines = {
    'A': 10, 'B': 15, 'C': 12, 
    'D': 20, 'E': 25, 'Goal': 40
}

def delivery_heuristic(current_node, target_node, time_elapsed):
    base_h = 10 
    time_remaining = deadlines.get(current_node, 100) - time_elapsed
    return base_h + time_remaining

def optimize_delivery(map_network, start, end):
    queue = [(0, start, [start], 0)]
    visited = set()

    print(f"Starting delivery route from {start} to {end}...")

    while queue:
        priority, current, path, current_time = heapq.heappop(queue)

        if current == end:
            print(f"\nFinal Route Found: {' -> '.join(path)}")
            print(f"Total Time: {current_time}")
            return path

        if current not in visited:
            visited.add(current)
            
            for neighbor, travel_time in map_network[current].items():
                if neighbor not in visited:
                    arrival_time = current_time + travel_time
                    
                    if arrival_time <= deadlines.get(neighbor, float('inf')):
                        h_val = delivery_heuristic(neighbor, end, arrival_time)
                        heapq.heappush(queue, (h_val, neighbor, path + [neighbor], arrival_time))
                    else:
                        print(f"Skipping {neighbor}: Deadline would be missed.")

    print("No valid route found within time windows.")
    return None

optimize_delivery(delivery_points, 'Start', 'Goal')

Starting delivery route from Start to Goal...

Final Route Found: Start -> A -> D -> C -> Goal
Total Time: 17


['Start', 'A', 'D', 'C', 'Goal']